# Парсинг текстов

In [1]:
%%capture
!pip install selenium -q
!apt-get update -q
!apt-get install -y chromium-browser chromium-chromedriver -q
!pip install google_colab_selenium -q

In [2]:
import re
import requests
import time
import datetime
import pandas as pd
import warnings

from tqdm import tqdm
from bs4 import BeautifulSoup
from selenium import webdriver
from dataclasses import dataclass
from datetime import datetime, timedelta
from IPython import display

warnings.filterwarnings("ignore", category=FutureWarning)

In [3]:
import google_colab_selenium as gs
driver = gs.Chrome()

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [4]:
@dataclass
class Article:
    id: str = None
    url: str = None
    title: str = None
    subtitle: str = None
    content: str = None
    datetime: str = None

### РИА Новости

In [5]:
SLEEP = 2
DEPTH = 30
BASE_URL = "https://ria.ru/"
TOPICS = ["economy", "society", "science", "sport", "tourism"]

Функция для сбора данных страниц со статьями

In [6]:
def get_pages():

    """Load and scroll pages"""

    items, topics_order = [], []

    for topic in TOPICS:
        try:
            old_size = len(items)
            URL = BASE_URL + topic
            driver.get(URL)
            time.sleep(SLEEP)

            # push to list 20 next articles
            driver.execute_script(
                "document.getElementsByClassName('list-more')[0].click()"
            )
            time.sleep(1)

            # scroll page to automatically load more articles
            for i in tqdm(range(DEPTH), leave=False):
                try:
                    driver.execute_script(
                        f"window.scrollTo(0, document.body.scrollHeight - 1200)"
                    )
                    time.sleep(1)
                except:
                    pass

            # find all pages
            html = driver.page_source
            soup = BeautifulSoup(html, "html.parser")
            scope = soup.find(
                "div", {"class": "list", "itemtype": "http://schema.org/ItemList"}
            )
            items += scope.find_all("div", {"class": "list-item"})

            # number of pages can not be multiple of deepth*20
            # that's why we count topics_order dynamically
            new_size = len(items)
            if new_size > old_size:
                topics_order.extend([topic] * (new_size - old_size))
        except:
            pass

    return items, topics_order

Функция для парсинга контента со страниц

In [7]:
def parse_page(page):
    """Extract from page desired fields"""

    # Create article data class object
    article = Article()

    # article url
    article.url = page.find("a", {"class": "list-item__image"})["href"]

    # article id
    s = re.findall(r"\d+.html", article.url)[0]
    article.id = s[: s.find(".")]

    # load page
    driver.get(article.url)
    time.sleep(SLEEP)
    html = driver.page_source

    # article source
    source = article.url[8 : article.url.find(".")]

    # article object
    soup = BeautifulSoup(html, "html.parser")
    obj = soup.find(
        "div",
        {
            "class": lambda x: x and (x.find(f"article m-article m-{source}") > -1),
            "data-article-id": article.id,
        },
    )

    if not obj:
        obj = soup.find(
            "div",
            {
                "class": lambda x: x and (x.find(f"article m-video m-{source}") > -1),
                "data-article-id": article.id,
            },
        )

    # title
    title = obj.find("div", {"class": "article__title"})
    title_2 = obj.find("h1", {"class": "article__title"})

    if title:
        article.title = title.text
    else:
        article.title = title_2.text if title_2 else ""

    # subtitle
    subtitle = obj.find("h1", {"class": "article__second-title"})
    article.subtitle = subtitle.text if subtitle else ""

    # content
    article.content = obj.find(
        "div", {"class": "article__body js-mediator-article mia-analytics"}
    ).text

    # datetime
    article.datetime = obj.find("div", {"class": "article__info-date"}).find("a").text

    return article

In [8]:
# pages, topics_order = get_pages()
# len(pages)

In [9]:
# data, topics_order_fixed = [], []

# for num, page in enumerate(tqdm(pages)):
#     try:
#         res = parse_page(page)

#         data.append(res)
#         topics_order_fixed.append(topics_order[num])
#     except:
#         pass

In [10]:
# df = pd.DataFrame(data=data)

# df.info()

In [11]:
# df["topic"] = topics_order_fixed

In [12]:
# df.head()

In [13]:
# df.to_csv('ria_news.csv')

### Lenta.ru

In [14]:
class lentaRu_parser:
    def __init__(self):
        pass

    def _get_url(self, param_dict: dict) -> str:
        """
        Возвращает URL для запроса json таблицы со статьями
        """
        hasType = int(param_dict['type']) != 0
        hasBloc = int(param_dict['bloc']) != 0

        url = (
            'https://lenta.ru/search/v2/process?'
            + 'from={}&'.format(param_dict['from'])
            + 'size={}&'.format(param_dict['size'])
            + 'sort={}&'.format(param_dict['sort'])
            + 'title_only={}&'.format(param_dict['title_only'])
            + 'domain={}&'.format(param_dict['domain'])
            + 'modified%2Cformat=yyyy-MM-dd&'
        )

        # Добавляем условные параметры только если они нужны
        if hasType:
            url += 'type={}&'.format(param_dict['type'])
        if hasBloc:
            url += 'bloc={}&'.format(param_dict['bloc'])

        url += (
            'modified%2Cfrom={}&'.format(param_dict['dateFrom'])
            + 'modified%2Cto={}&'.format(param_dict['dateTo'])
            + 'query={}'.format(param_dict['query'])
        )

        return url

    def _get_search_table(self, param_dict: dict) -> pd.DataFrame:
        """
        Возвращает pd.DataFrame со списком статей
        """
        url = self._get_url(param_dict)
        r = requests.get(url)
        r.raise_for_status()  # полезно для явной обработки ошибок
        search_table = pd.DataFrame(r.json()['matches'])
        return search_table

    def get_articles(
        self,
        param_dict,
        time_step=37,
        save_every=5
    ) -> pd.DataFrame:
        """
        Функция для скачивания статей интервалами через каждые time_step дней
        Делает сохранение таблицы через каждые save_every * time_step дней
        """
        param_copy = param_dict.copy()
        time_step = timedelta(days=time_step)
        dateFrom = datetime.strptime(param_copy['dateFrom'], '%Y-%m-%d')
        dateTo = datetime.strptime(param_copy['dateTo'], '%Y-%m-%d')
        if dateFrom > dateTo:
            raise ValueError('dateFrom should be less than dateTo')

        out = pd.DataFrame()
        save_counter = 0

        while dateFrom <= dateTo:
            param_copy['dateTo'] = (dateFrom + time_step).strftime('%Y-%m-%d')
            if dateFrom + time_step > dateTo:
                param_copy['dateTo'] = dateTo.strftime('%Y-%m-%d')

            print(
                'Parsing articles from '
                + param_copy['dateFrom'] + ' to ' + param_copy['dateTo']
            )

            chunk_df = self._get_search_table(param_copy)

            out = pd.concat([out, chunk_df], ignore_index=True)

            dateFrom += time_step + timedelta(days=1)
            param_copy['dateFrom'] = dateFrom.strftime('%Y-%m-%d')
            save_counter += 1

            if save_counter == save_every:
                display.clear_output(wait=True)
                out.to_excel("/tmp/checkpoint_table.xlsx", index=False)
                print('Checkpoint saved!')
                save_counter = 0

        print('Finish')

        out = out.drop(columns=['modified', 'lastmodtime', 'type',
                                'domain', 'status', 'part',
                                'tags', 'image_url', 'rightcol'])
        out.rename(columns={'snippet': 'subtitle', 'docid': 'id', 'text': 'content', 'pubdate': 'datetime'},
                   inplace=True)

        return out

In [15]:
parser = lentaRu_parser()

In [16]:
query = ''
offset = 0
size = 1000
sort = "3"
title_only = "0"
domain = "1"
material = "0"
bloc = "0" # topic = тематика новости
dateFrom = '2023-01-01'
dateTo = "2025-12-17"

lenta_blocks = {'Общество': 1,
                'Экономика': 4,
                'Силовые структуры': 37,
                'Бывший СССР': 3,
                'Спорт': 8,
                'Забота о себе': 87,
                'Строительство': 0,
                'Туризм/Путешествия': 48,
                'Наука и техника': 5,}

In [17]:
# lenta_df = pd.DataFrame(columns=['id', 'url', 'title', 'bloc', 'datetime', 'content', 'subtitle'])

# for query in lenta_blocks.keys():

#     param_dict = {'query'     : query.lower(),
#                   'from'      : str(offset),
#                   'size'      : str(size),
#                   'dateFrom'  : dateFrom,
#                   'dateTo'    : dateTo,
#                   'sort'      : sort,
#                   'title_only': title_only,
#                   'type'      : material,
#                   'bloc'      : bloc,
#                   'domain'    : domain}

#     tbl = parser.get_articles(param_dict=param_dict,
#                            time_step=37,
#                            save_every=5)


#     lenta_df = pd.concat([lenta_df, tbl], ignore_index=True)

# len(lenta_df)

In [18]:
# lenta_df.to_csv('lenta_news.csv')

#  Первичная обработка и разметка тем

### РИА Новости

In [43]:
ria_df = pd.read_csv('ria_news.csv').drop(columns=['Unnamed: 0'])

In [44]:
ria_df['topic'].value_counts()

,count
topic,
science,624
society,379
economy,375


In [45]:
topic_mapping = {
    'science': 8,
    'society': 0,
    'economy': 1
}

ria_df['topic'] = ria_df['topic'].map(topic_mapping)

In [46]:
ria_df['content'].sample(5)

,content
1252,"МОСКВА, 19 сен – РИА Новости. Куликовская битв..."
1089,"МОСКВА, 18 окт - РИА Новости. Значительное уве..."
49,"МОСКВА, 16 дек - РИА Новости. Иск Банка России..."
1166,"МОСКВА, 7 окт — РИА Новости. Нобелевскую преми..."
1100,"МОСКВА, 17 окт – РИА Новости. Два ИТ-решения ""..."


Удалим шапку

In [47]:
pattern = r'^[А-ЯЁ]+(?:-[А-ЯЁ]+)?,\s*\d{1,2}\s+[а-яё]+\.?\s*[–\-—]\s*РИА Новости\.?\s*'
ria_df['content'] = ria_df['content'].apply(lambda x: re.sub(pattern, '', x))
ria_df['content'].sample(5)

,content
1372,Грузовой космический корабль Dragon американск...
42,Еврокомиссия в рамках опубликованного пакета м...
463,"В салонах автобусов ""Мострансавто"" начали звуч..."
1359,"Противоударный учебный беспилотник, оснащенный..."
886,"Новый материал, который работает как ловушка д..."


In [48]:
ria_df['content'].str[-20:].sample(5)

,content
973,науки и технологий.
1266,18:43\n\n\nПоделиться\n\n
1352,"ств"", – заключил он."
438,21:43\n\n\nПоделиться\n\n
1162,07:32\n\n\nПоделиться\n\n


In [49]:
ria_df['content'] = ria_df['content'].str.replace('Поделиться', '', regex=False)

In [68]:
ria_df.shape

(1378, 7)

### Lenta.ru

In [58]:
lenta_df = pd.read_csv('lenta_news.csv').drop(columns=['Unnamed: 0'])

In [59]:
lenta_df['bloc'].value_counts()

,count
bloc,
4,12788
1,8652
2,5143
8,4864
3,4520
37,2567
12,1340
7,1193
5,1190


Замапим номера тем API Ленты и нашего соревнования

In [60]:
mapping_dict = {1: 0, 3: 3, 4: 1, 5: 8, 8: 4, 37: 2, 48: 7, 87: 5}

lenta_df['topic'] = lenta_df['bloc'].map(mapping_dict)

Так как у ленты нет отдельной рубрики под тему Строительство, разметим её по ключевым словам

In [61]:
construction_keywords = [
    'стройка',
    'строительные работы',

    'застройщик',
    'застройка',
    'стройплощадка',
    'строительная площадка',
    'генподрядчик',

    'возведение здания',
    'возведение объекта',
    'сдача объекта',
    'ввод в эксплуатацию',

    'строительные материалы',
    'стройматериалы',
    'цемент',
    'бетон',
    'кирпич',

    'строительная техника',
    'строительный кран',
    'экскаватор',
    'бульдозер',

    'строитель',
    'прораб',
    'инженер-строитель',
    'архитектор',
]

pattern = re.compile('|'.join(construction_keywords), re.IGNORECASE)
lenta_df['topic'] = lenta_df.apply(
    lambda row: 6 if pd.isna(row['topic']) and bool(pattern.search(str(row['content'])))
                  else row['topic'],
    axis=1
)

In [62]:
lenta_df['topic'].value_counts()

,count
topic,
1.0,12788
0.0,8652
4.0,4864
3.0,4520
6.0,3402
2.0,2567
8.0,1190
5.0,1164
7.0,636


Уберем статьи, не попавшие ни под одну целевую тему

In [66]:
lenta_df = lenta_df.dropna(subset=['topic'])
lenta_df = lenta_df.drop(columns=['bloc'])

In [67]:
lenta_df.shape

(39783, 7)

# Итоговый датасет

In [97]:
df = pd.concat([ria_df, lenta_df], ignore_index=True)
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 41161 entries, 0 to 41160
Data columns (total 7 columns):
 #   Column    Non-Null Count  Dtype  
---  ------    --------------  -----  
 0   id        41161 non-null  int64  
 1   url       41161 non-null  object 
 2   title     41161 non-null  object 
 3   subtitle  40946 non-null  object 
 4   content   41158 non-null  object 
 5   datetime  41161 non-null  object 
 6   topic     41161 non-null  float64
dtypes: float64(1), int64(1), object(5)
memory usage: 2.2+ MB


In [98]:
df.sample(5)

,id,url,title,subtitle,content,datetime,topic
21133,1777615,https://lenta.ru/news/2025/02/06/vsu-poteryali...,ВСУ потеряли крупную группировку войск в ДНР,Об этом сообщает ТАСС со ссылкой на российские...,Фото: Reuters Александр Курбатов Вооруженные с...,1738830900,3.0
12687,1609675,https://lenta.ru/news/2024/04/19/v-mvf-nazvali...,В МВФ назвали причины устойчивости экономики Р...,Фото: Sergei Karpukhin / Reuters Дмитрий Ворон...,Фото: Sergei Karpukhin / Reuters Дмитрий Ворон...,1713544500,1.0
34575,1575136,https://lenta.ru/news/2024/02/19/stroy-rating/,Названы города с наибольшим объемом строящегос...,Фото: Виталий Тимкив / РИА Новости Виктория Кл...,Фото: Виталий Тимкив / РИА Новости Виктория Кл...,1708326548,6.0
4158,1605338,https://lenta.ru/news/2024/04/12/v-rpts-prokom...,В РПЦ прокомментировали обвинения Эстонии в те...,Так его прокомментировал глава ... отдела МП п...,Фото: Мария Девахина / РИА Новости Кристина Ес...,1712914740,3.0
34768,1588397,https://lenta.ru/news/2024/03/14/stroy-dom/,Россияне стали чаще интересоваться одним видом...,Фото: Bulkin Sergey / Globallookpress.com Викт...,Фото: Bulkin Sergey / Globallookpress.com Викт...,1710411060,6.0


In [99]:
df['topic'].value_counts(normalize=True)

,proportion
topic,
1.0,0.319793
0.0,0.219407
4.0,0.118170
3.0,0.109813
6.0,0.082651
2.0,0.062365
8.0,0.044071
5.0,0.028279
7.0,0.015452


In [80]:
df.to_csv('dataset.csv', index=False)

In [100]:
df.to_csv('dataset_1.csv', index=False)

In [101]:
df.to_csv('dataset_2.csv')